# QIntent Structured Semantic Score for qLDPC-Style Post-Decoding

This notebook demonstrates a public-preview QIntent workflow for ranking qLDPC-style correction candidates after a decoder has produced candidate hypotheses.

The goal is not to claim that QIntent replaces a quantum error-correction decoder. Instead, this demo shows how QDSV can represent post-decoding candidates as structured decision states and rank them using prepared evidence blocks such as syndrome support, logical safety, decoder confidence, propagation safety, and risk.

The internal QDSV scoring formula is not exposed. The notebook only exposes the public declaration, evidence fields, block-level trace, and ranking output.

## 1. Install the SDK from GitHub main

The structured semantic score operation is installed from the public GitHub repository so this notebook can run before the next PyPI release.

In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"

In [ ]:
import json
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

spec = client.spec()
spec["status"]

## 2. Prepare qLDPC-style candidate evidence

Each row represents a candidate correction hypothesis after a decoder or predecoder has produced candidate paths. Values are prepared on a 0..1000 operational scale before entering QIntent.

This is a toy public-safe dataset, not a hardware qLDPC benchmark yet.

In [ ]:
rows = [
    {
        "candidate_index": 0,
        "syndrome_support": 930,
        "check_consistency": 900,
        "logical_preservation": 820,
        "distance_safety": 830,
        "decoder_confidence": 850,
        "propagation_safety": 820,
        "syndrome_risk": 50,
        "logical_risk": 120,
        "syndrome_entropy_adjustment": -10,
    },
    {
        "candidate_index": 1,
        "syndrome_support": 940,
        "check_consistency": 880,
        "logical_preservation": 720,
        "distance_safety": 700,
        "decoder_confidence": 870,
        "propagation_safety": 760,
        "syndrome_risk": 80,
        "logical_risk": 260,
        "syndrome_entropy_adjustment": -20,
    },
    {
        "candidate_index": 2,
        "syndrome_support": 890,
        "check_consistency": 910,
        "logical_preservation": 930,
        "distance_safety": 910,
        "decoder_confidence": 830,
        "propagation_safety": 890,
        "syndrome_risk": 30,
        "logical_risk": 40,
        "syndrome_entropy_adjustment": 15,
    },
]

df = pd.DataFrame(rows)
df

## 3. Baseline: decoder-confidence-only ranking

A simple post-decoding policy could choose the candidate with the highest decoder confidence. This baseline ignores logical risk, distance safety, propagation safety, and syndrome-level adjustments.

In [ ]:
baseline = df.sort_values("decoder_confidence", ascending=False).reset_index(drop=True)
baseline[["candidate_index", "decoder_confidence", "logical_risk", "logical_preservation", "distance_safety"]]

## 4. QIntent structured semantic score declaration

The QIntent source below declares public evidence blocks. It does not expose the private QDSV formula.

The three blocks are:

- `syndrome`: syndrome support and check consistency.
- `logical_safety`: logical preservation and distance safety.
- `decoder`: decoder confidence and propagation safety.

In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_post_decoding")
  .accept_if(threshold=600)
  .rank()
  .top_k(2)
"""

print(source)

## 5. Explain before running

`explain()` returns a semantic execution passport. For this operation, it should identify structured semantic scoring, block count, signal count, and confirm that the internal formula is not exposed.

In [ ]:
passport = client.explain(source, rows=rows)
predicate_summary = passport["semantic_execution_passport"]["predicate"]
predicate_summary

Expected public evidence:

- `kind`: `structured_semantic_score`
- `blocks_count`: 3
- `signals_count`: 6
- `internal_formula_exposed`: `False`

In [ ]:
assert predicate_summary["kind"] == "structured_semantic_score"
assert predicate_summary["blocks_count"] == 3
assert predicate_summary["signals_count"] == 6
assert predicate_summary["internal_formula_exposed"] is False

predicate_summary

## 6. Run QIntent and inspect ranked correction candidates

In [ ]:
result = client.run(source, rows=rows)
result["status"]

In [ ]:
selected_rows = result["result"]["selected_rows"]
ranked = pd.DataFrame(selected_rows)
ranked[[
    "candidate_index",
    "_qdsv_rank",
    "_qdsv_structured_semantic_score",
    "decoder_confidence",
    "logical_preservation",
    "distance_safety",
    "logical_risk",
]]

## 7. Compare baseline vs QDSV structured ranking

The baseline chooses by decoder confidence alone. QDSV ranks candidates using structured evidence and risk. In this toy example, the top QDSV candidate is not the same as the highest-confidence decoder candidate.

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "decoder_confidence_only",
        "top_candidate": int(baseline.iloc[0]["candidate_index"]),
        "decoder_confidence": int(baseline.iloc[0]["decoder_confidence"]),
        "logical_risk": int(baseline.iloc[0]["logical_risk"]),
    },
    {
        "method": "qdsv_structured_semantic_score",
        "top_candidate": int(ranked.iloc[0]["candidate_index"]),
        "decoder_confidence": int(ranked.iloc[0]["decoder_confidence"]),
        "logical_risk": int(ranked.iloc[0]["logical_risk"]),
    },
])

comparison

## 8. Inspect the audit trace for the top candidate

The selected row includes a public audit trace with block-level values, risk fields, selected status, and rank. The private formula remains hidden.

In [ ]:
top_trace = selected_rows[0]["_qdsv_decision_model"]
print(json.dumps(top_trace, indent=2))

## 9. Save evidence artifacts

The JSON evidence file can be attached to a review pack, reproduced later, or compared with future experiments using real qLDPC decoders.

In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_post_decoding",
    "rows": rows,
    "qintent_source": source,
    "predicate_summary": predicate_summary,
    "baseline_top_candidate": int(baseline.iloc[0]["candidate_index"]),
    "qdsv_top_candidate": int(ranked.iloc[0]["candidate_index"]),
    "comparison": comparison.to_dict(orient="records"),
    "selected_rows": selected_rows,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_structured_score_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
ranked.to_csv("qdsv_qldpc_structured_score_selected_rows.csv", index=False)
comparison.to_csv("qdsv_qldpc_structured_score_comparison.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_structured_score_evidence.json")
print("- qdsv_qldpc_structured_score_selected_rows.csv")
print("- qdsv_qldpc_structured_score_comparison.csv")

## Interpretation

This notebook demonstrates the post-decoding role of QDSV/QIntent: given candidate correction hypotheses, QDSV represents the candidates as structured decision states and QIntent executes a public structured scoring declaration over prepared evidence.

The next experimental step is to replace the toy rows with candidates generated by an actual qLDPC decoder or predecoder and evaluate whether QDSV improves ranking stability, logical-risk avoidance, auditability, or ambiguity resolution.